# Preprocessing

In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from pathlib import Path
from src.aggregate import aggregate

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")

In [2]:
pos = pd.read_csv(RAW / "POS_CASH_balance.csv")
pos.shape

(10001358, 8)

# Feature Engineering

## Iteration 1

In [3]:
p = aggregate(pos, "SK_ID_CURR", "pos", cat_cols=["NAME_CONTRACT_STATUS"], ignore=["SK_ID_PREV"])

In [4]:
pos = pos.sort_values(["SK_ID_PREV", "MONTHS_BALANCE"]).reset_index(drop=True)
pos["is_dpd"] = (pos["SK_DPD"] > 0).astype("int8")
gp = pos.groupby("SK_ID_PREV")
loan = pd.DataFrame({
    "curr": gp["SK_ID_CURR"].first(),
    "max_dpd": gp["SK_DPD"].max(),
    "completed": (gp["NAME_CONTRACT_STATUS"].last() == "Completed").astype("int8"),
    "future_last": gp["CNT_INSTALMENT_FUTURE"].last(),
    "term": gp["CNT_INSTALMENT"].max(),
})
pos["blk"] = (pos["SK_ID_PREV"].ne(pos["SK_ID_PREV"].shift()) | pos["is_dpd"].ne(pos["is_dpd"].shift())).cumsum()
runs = pos[pos["is_dpd"] == 1].groupby(["SK_ID_PREV", "blk"]).size()
loan["streak"] = runs.groupby("SK_ID_PREV").max()
loan["streak"] = loan["streak"].fillna(0)
loan["progress_remaining"] = (loan["future_last"] / loan["term"]).replace([np.inf, -np.inf], np.nan)
gc = loan.groupby("curr")
p = p.join(pd.DataFrame({
    "pos_n_loans": gc.size(),
    "pos_max_dpd_streak": gc["streak"].max(),
    "pos_worst_loan_dpd": gc["max_dpd"].max(),
    "pos_completed_share": gc["completed"].mean(),
    "pos_progress_remaining_mean": gc["progress_remaining"].mean(),
}))

In [5]:
distressed = ["Demand", "Returned to the store", "Amortized debt", "Canceled"]
p = p.join(pd.DataFrame({
    "pos_dpd_month_share": (pos["SK_DPD"] > 0).groupby(pos["SK_ID_CURR"]).mean(),
    "pos_dpd_def_month_share": (pos["SK_DPD_DEF"] > 0).groupby(pos["SK_ID_CURR"]).mean(),
    "pos_distressed_share": pos["NAME_CONTRACT_STATUS"].isin(distressed).groupby(pos["SK_ID_CURR"]).mean(),
}))
p.shape

(337252, 52)

In [6]:
parts = []
for m in [6, 12, 24, 60]:
    w = pos[pos["MONTHS_BALANCE"] >= -m].groupby("SK_ID_CURR")
    parts += [w["SK_DPD"].max().rename(f"pos_dpd_max_{m}"),
              w["SK_DPD"].mean().rename(f"pos_dpd_mean_{m}"),
              w["is_dpd"].mean().rename(f"pos_late_share_{m}"),
              w["SK_DPD_DEF"].mean().rename(f"pos_dpd_def_mean_{m}")]
windows = pd.concat(parts, axis=1)

frac = {}
for s, l in [(6, 12), (12, 24), (24, 60)]:
    frac[f"pos_late_share_frac_{s}_{l}"] = windows[f"pos_late_share_{s}"] / windows[f"pos_late_share_{l}"]
frac = pd.DataFrame(frac).replace([np.inf, -np.inf], np.nan)

g = pos.groupby("SK_ID_CURR")
dx = pos["MONTHS_BALANCE"] - g["MONTHS_BALANCE"].transform("mean")
dy = pos["SK_DPD"] - g["SK_DPD"].transform("mean")
trend = (dx * dy).groupby(pos["SK_ID_CURR"]).sum() / (dx * dx).groupby(pos["SK_ID_CURR"]).sum()

recent = pos.sort_values("MONTHS_BALANCE").groupby("SK_ID_CURR").tail(1)[["SK_ID_CURR", "SK_ID_PREV"]]
ll = pos.merge(recent, on=["SK_ID_CURR", "SK_ID_PREV"]).groupby("SK_ID_CURR")
lastloan = pd.DataFrame({"pos_last_loan_dpd_max": ll["SK_DPD"].max(),
                         "pos_last_loan_dpd_mean": ll["SK_DPD"].mean(),
                         "pos_last_loan_late_share": ll["is_dpd"].mean()})

p = p.join(windows).join(frac).join(trend.replace([np.inf, -np.inf], np.nan).rename("pos_dpd_trend")).join(lastloan)
p.shape

(337252, 75)

## Iteration 2

In [7]:
for status, tag in [("Active", "active"), ("Completed", "done")]:
    sub = pos[pos["NAME_CONTRACT_STATUS"] == status]
    a = aggregate(sub, "SK_ID_CURR", f"pos_{tag}", ignore=["SK_ID_PREV", "blk"])
    p = p.join(a)
p = p.replace([np.inf, -np.inf], np.nan)
p.shape

(337252, 137)

# Save

In [8]:
p.reset_index().to_pickle(INTERIM / "pos_cash_balance.pkl")
p.shape

(337252, 137)